# Man vs Machine on Oslo Børs — Strategy Evaluation

This notebook evaluates the backtested portfolio returns from the Simulation notebook against standard asset-pricing benchmarks, for both the historical and modern tracks:

- **Risk-adjusted performance:** annualized return, volatility, Sharpe ratio, and maximum drawdown.
- **Alpha regressions:** CAPM, Fama-French 3-factor, and Carhart 4-factor regressions to test whether returns are genuine alpha or just compensation for known risk factors.
- **Factor loadings:** a detailed breakdown of exposure to Market, SMB, HML, and UMD for the flagship strategies.
- **Portfolio characteristics:** average stock-level features (size, momentum, volatility, fund conviction) held in the Long vs. Short baskets, for appendix reporting.

**Input:** `portfolio_simulation_returns_mod.csv`, `portfolio_simulation_returns_hist.csv`, `Norway_Rf_monthly.csv`, `Norway_market_portfolios_monthly.csv`, `Norway_pricing_factors_monthly.csv`, `modern_equity_dataset.csv`, `investable_universe.csv`, `modern_predictions.csv`, `historical_predictions.csv`
**Output:** risk, alpha, and factor-loading tables; portfolio characteristics appendix tables; a combined headline results table.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Setup

The simulated portfolio returns are merged with Ødegaard's risk-free rate, market portfolio, and Fama-French/Carhart pricing factors, all aligned to month-end. This gives a single evaluation matrix per track with everything needed for the risk and alpha regressions below.

In [2]:
print("Loading and aligning datasets...")

# Simulated portfolio returns (from the Simulation notebook)
sim_returns_mod = pd.read_csv('../Data/portfolio_simulation_returns_mod.csv')
sim_returns_mod['TradeDate'] = pd.to_datetime(sim_returns_mod['TradeDate']) + pd.offsets.MonthEnd(0)

sim_returns_hist = pd.read_csv('../Data/portfolio_simulation_returns_hist.csv')
sim_returns_hist['TradeDate'] = pd.to_datetime(sim_returns_hist['TradeDate']) + pd.offsets.MonthEnd(0)

# Ødegaard's risk-free rate (skiprows=1 bypasses the text header)
rf_df = pd.read_csv('../Data/Norway_Rf_monthly.csv', skiprows=1)
rf_df['TradeDate'] = pd.to_datetime(rf_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
rf_df = rf_df[['TradeDate', 'Rf(1m)']]

# Ødegaard's market portfolios (value-weighted 'VW')
mkt_df = pd.read_csv('../Data/Norway_market_portfolios_monthly.csv')
mkt_df['TradeDate'] = pd.to_datetime(mkt_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
mkt_df = mkt_df[['TradeDate', 'VW']]

# Ødegaard's Fama-French / Carhart pricing factors
ff_df = pd.read_csv('../Data/Norway_pricing_factors_monthly.csv')
ff_df['TradeDate'] = pd.to_datetime(ff_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
ff_df = ff_df[['TradeDate', 'SMB', 'HML', 'UMD']]

# Master merge
eval_df_mod = sim_returns_mod.merge(rf_df, on='TradeDate', how='left')
eval_df_mod = eval_df_mod.merge(mkt_df, on='TradeDate', how='left')
eval_df_mod = eval_df_mod.merge(ff_df, on='TradeDate', how='left')

eval_df_hist = sim_returns_hist.merge(rf_df, on='TradeDate', how='left')
eval_df_hist = eval_df_hist.merge(mkt_df, on='TradeDate', how='left')
eval_df_hist = eval_df_hist.merge(ff_df, on='TradeDate', how='left')

# Drop rows where Ødegaard's data is missing at the edges of the sample
eval_df_mod = eval_df_mod.dropna().reset_index(drop=True)
eval_df_hist = eval_df_hist.dropna().reset_index(drop=True)

# Market risk premium (Rm - Rf)
eval_df_mod['Mkt-RF'] = eval_df_mod['VW'] - eval_df_mod['Rf(1m)']
eval_df_hist['Mkt-RF'] = eval_df_hist['VW'] - eval_df_hist['Rf(1m)']

# Attach the momentum baseline to the historical track (only computed on the modern-track dates)
eval_df_hist['Net_Ret_Mom_LS'] = eval_df_mod['Net_Ret_Mom_LS']

# Strategies to evaluate on each track
strategy_cols_mod = [
    'Net_Ret_LongOnly',       # Mutual Fund AI strategy
    'Net_Ret_LongShort',      # Hedge Fund AI strategy (Ensemble)
    'Net_Ret_Modern_XGB_LS',  # XGBoost Hedge Fund strategy
    'Net_Ret_Mom_LS',         # Traditional finance baseline
    'VW'                      # Overall market
]

strategy_cols_hist = [
    'Net_Ret_LongOnly',
    'Net_Ret_LongShort',
    'Net_Ret_Hist_XGB_LS',
    'Net_Ret_Mom_LS',
    'VW'
]

print(f"Data successfully aligned. Total valid months: {eval_df_mod.shape[0]}")
print("\nPreview of the evaluation matrix:")
display(eval_df_mod[['TradeDate'] + strategy_cols_mod + ['Rf(1m)', 'Mkt-RF', 'SMB', 'HML', 'UMD']].head())

Loading and aligning datasets...
Data successfully aligned. Total valid months: 35

Preview of the evaluation matrix:


,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Modern_XGB_LS,Net_Ret_Mom_LS,VW,Rf(1m),Mkt-RF,SMB,HML,UMD
0,2022-01-31,0.012170,0.069049,0.071368,0.084627,-0.020529,0.00073,-0.021259,-0.029103,0.107942,0.027788
1,2022-02-28,0.030086,-0.057215,-0.028577,-0.028051,0.028558,0.00074,0.027818,-0.072052,0.152077,0.065305
2,2022-03-31,0.010532,0.079223,0.048805,0.009001,0.065944,0.00087,0.065074,0.038270,0.077338,0.063125
3,2022-04-30,0.074331,0.033882,0.056048,0.045937,-0.018648,0.00081,-0.019458,-0.040243,0.096603,0.009725
4,2022-05-31,-0.057338,0.029581,0.003516,-0.003264,0.052633,0.00083,0.051803,0.078451,0.034903,0.068843


In [3]:
display(eval_df_hist)
display(eval_df_mod)

,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Benchmark,Turnover_LongOnly,Turnover_LongShort,Net_Ret_Hist_XGB_LS,Rf(1m),VW,SMB,HML,UMD,Mkt-RF,Net_Ret_Mom_LS
0,2022-01-31,-0.005068,0.062356,-0.021959,1.000000,2.000000,0.045873,0.00073,-0.020529,-0.029103,0.107942,0.027788,-0.021259,0.084627
1,2022-02-28,0.070887,-0.020483,0.073510,0.769231,1.384615,-0.019364,0.00074,0.028558,-0.072052,0.152077,0.065305,0.027818,-0.028051
2,2022-03-31,0.014730,0.088868,-0.019149,0.792453,1.396226,0.092025,0.00087,0.065944,0.038270,0.077338,0.063125,0.065074,0.009001
3,2022-04-30,0.132280,0.106421,0.052168,0.925926,1.629630,0.087618,0.00081,-0.018648,-0.040243,0.096603,0.009725,-0.019458,0.045937
4,2022-05-31,-0.046850,0.051578,-0.066441,0.785714,1.428571,0.059913,0.00083,0.052633,0.078451,0.034903,0.068843,0.051803,-0.003264
5,2022-06-30,-0.005758,-0.001389,0.018326,0.642857,1.142857,0.004937,0.00114,-0.087680,0.008334,-0.000904,0.043376,-0.088820,-0.023316
6,2022-07-31,0.065830,0.089541,0.017478,0.758621,1.344828,0.070496,0.00135,0.056899,0.006467,-0.008872,-0.025862,0.055549,0.082034
7,2022-08-31,-0.085546,0.077864,-0.125781,0.586207,1.103448,0.059071,0.00169,0.013373,0.000874,0.090788,0.102345,0.011683,0.042863
8,2022-09-30,0.074608,0.071769,0.017229,0.745763,1.525424,0.071548,0.00225,-0.107123,-0.014143,0.040670,0.050118,-0.109373,0.071689
9,2022-10-31,0.033976,0.052415,0.020553,0.779661,1.661017,0.037245,0.00228,0.091229,0.026409,0.130273,0.001536,0.088949,0.004392


,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Benchmark,Turnover_LongOnly,Turnover_LongShort,Net_Ret_Modern_XGB_LS,Net_Ret_Mom_LS,Rf(1m),VW,SMB,HML,UMD,Mkt-RF
0,2022-01-31,0.012170,0.069049,-0.021959,1.000000,2.000000,0.071368,0.084627,0.00073,-0.020529,-0.029103,0.107942,0.027788,-0.021259
1,2022-02-28,0.030086,-0.057215,0.073510,0.769231,1.461538,-0.028577,-0.028051,0.00074,0.028558,-0.072052,0.152077,0.065305,0.027818
2,2022-03-31,0.010532,0.079223,-0.019149,0.754717,1.320755,0.048805,0.009001,0.00087,0.065944,0.038270,0.077338,0.063125,0.065074
3,2022-04-30,0.074331,0.033882,0.052168,1.000000,1.703704,0.056048,0.045937,0.00081,-0.018648,-0.040243,0.096603,0.009725,-0.019458
4,2022-05-31,-0.057338,0.029581,-0.066441,1.035714,1.714286,0.003516,-0.003264,0.00083,0.052633,0.078451,0.034903,0.068843,0.051803
5,2022-06-30,0.007678,0.013274,0.018326,0.928571,1.500000,-0.025950,-0.023316,0.00114,-0.087680,0.008334,-0.000904,0.043376,-0.088820
6,2022-07-31,0.057892,0.055103,0.017478,1.000000,1.655172,0.057757,0.082034,0.00135,0.056899,0.006467,-0.008872,-0.025862,0.055549
7,2022-08-31,-0.091426,0.079350,-0.125781,0.724138,1.241379,0.046567,0.042863,0.00169,0.013373,0.000874,0.090788,0.102345,0.011683
8,2022-09-30,0.053592,0.040663,0.017229,0.745763,1.525424,0.038567,0.071689,0.00225,-0.107123,-0.014143,0.040670,0.050118,-0.109373
9,2022-10-31,0.033591,0.053415,0.020553,0.779661,1.491525,0.035388,0.004392,0.00228,0.091229,0.026409,0.130273,0.001536,0.088949


## 2. Modern Track Evaluation

### 2.1 Risk-Adjusted Performance

In [4]:
print("Calculating risk metrics...\n")

risk_results_mod = []

for col in strategy_cols_mod:
    # Excess return (for VW this subtracts Rf to get its own Sharpe ratio)
    excess_return = eval_df_mod[col] - eval_df_mod['Rf(1m)']

    ann_ret = eval_df_mod[col].mean() * 12
    ann_vol = eval_df_mod[col].std() * np.sqrt(12)

    ann_excess_ret = excess_return.mean() * 12
    sharpe_ratio = ann_excess_ret / ann_vol if ann_vol != 0 else 0

    # Maximum drawdown from a cumulative wealth index starting at 1.0
    cum_wealth = (1 + eval_df_mod[col]).cumprod()
    running_max = cum_wealth.cummax()
    drawdowns = (cum_wealth - running_max) / running_max
    max_drawdown = drawdowns.min()

    risk_results_mod.append({
        'Strategy': col,
        'Ann_Return': ann_ret,
        'Ann_Volatility': ann_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown
    })

risk_df_mod = pd.DataFrame(risk_results_mod)
risk_df_mod_display = risk_df_mod.copy()
risk_df_mod_display['Ann_Return'] = risk_df_mod_display['Ann_Return'].map('{:.2%}'.format)
risk_df_mod_display['Ann_Volatility'] = risk_df_mod_display['Ann_Volatility'].map('{:.2%}'.format)
risk_df_mod_display['Sharpe_Ratio'] = risk_df_mod_display['Sharpe_Ratio'].map('{:.2f}'.format)
risk_df_mod_display['Max_Drawdown'] = risk_df_mod_display['Max_Drawdown'].map('{:.2%}'.format)

print("=== Risk-Adjusted Performance Metrics (Modern) ===")
display(risk_df_mod_display)

Calculating risk metrics...

=== Risk-Adjusted Performance Metrics (Modern) ===


,Strategy,Ann_Return,Ann_Volatility,Sharpe_Ratio,Max_Drawdown
0,Net_Ret_LongOnly,28.87%,29.30%,0.87,-9.14%
1,Net_Ret_LongShort,50.26%,31.64%,1.48,-22.30%
2,Net_Ret_Modern_XGB_LS,39.65%,27.81%,1.30,-11.36%
3,Net_Ret_Mom_LS,47.53%,25.73%,1.71,-7.32%
4,VW,8.41%,14.31%,0.35,-12.75%


### 2.2 Alpha Regressions

CAPM, Fama-French 3-factor, and Carhart 4-factor regressions are run on each strategy's excess return to test whether performance survives after controlling for standard risk factors.

In [5]:
print("Running CAPM, FF3, and Carhart 4-Factor regressions...\n")

strategies_to_test_mod = [
    'Net_Ret_LongOnly',
    'Net_Ret_LongShort',
    'Net_Ret_Modern_XGB_LS',
    'Net_Ret_Mom_LS'
]

alpha_results_mod = []

for strat in strategies_to_test_mod:
    Y = eval_df_mod[strat] - eval_df_mod['Rf(1m)']

    X_capm = sm.add_constant(eval_df_mod[['Mkt-RF']])
    capm_model = sm.OLS(Y, X_capm).fit()

    X_ff3 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML']])
    ff3_model = sm.OLS(Y, X_ff3).fit()

    X_ff4 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()

    # Alpha (intercept) is annualized (* 12); t-stat and p-value are also extracted
    alpha_results_mod.append({
        'Strategy': strat,
        'CAPM_Alpha': capm_model.params['const'] * 12,
        'CAPM_t': capm_model.tvalues['const'],
        'CAPM_p': capm_model.pvalues['const'],

        'FF3_Alpha': ff3_model.params['const'] * 12,
        'FF3_t': ff3_model.tvalues['const'],
        'FF3_p': ff3_model.pvalues['const'],

        'FF4_Alpha': ff4_model.params['const'] * 12,
        'FF4_t': ff4_model.tvalues['const'],
        'FF4_p': ff4_model.pvalues['const']
    })

alpha_df_mod = pd.DataFrame(alpha_results_mod)

def format_alpha(alpha, pval):
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    return f"{alpha:.2%}{stars}"

for model in ['CAPM', 'FF3', 'FF4']:
    alpha_df_mod[f'{model}_Alpha_Str'] = alpha_df_mod.apply(lambda row: format_alpha(row[f'{model}_Alpha'], row[f'{model}_p']), axis=1)
    alpha_df_mod[f'{model}_t'] = alpha_df_mod[f'{model}_t'].map('{:.2f}'.format)
    alpha_df_mod[f'{model}_p'] = alpha_df_mod[f'{model}_p'].map('{:.3f}'.format)

alpha_table_mod = alpha_df_mod[[
    'Strategy',
    'CAPM_Alpha_Str', 'FF3_Alpha_Str', 'FF4_Alpha_Str',
    'CAPM_t', 'FF3_t', 'FF4_t',
    'CAPM_p', 'FF3_p', 'FF4_p'
]].copy()

alpha_table_mod.columns = [
    'Strategy',
    'CAPM Alpha', 'FF3 Alpha', 'FF4 Alpha',
    'CAPM t-stat', 'FF3 t-stat', 'FF4 t-stat',
    'CAPM p-val', 'FF3 p-val', 'FF4 p-val'
]

print("=== Combined Alpha Summary Table (Modern) ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
display(alpha_table_mod)

Running CAPM, FF3, and Carhart 4-Factor regressions...

=== Combined Alpha Summary Table (Modern) ===
Note: *** p<0.01, ** p<0.05, * p<0.10



,Strategy,CAPM Alpha,FF3 Alpha,FF4 Alpha,CAPM t-stat,FF3 t-stat,FF4 t-stat,CAPM p-val,FF3 p-val,FF4 p-val
0,Net_Ret_LongOnly,26.94%,29.25%**,31.42%**,1.56,2.06,2.18,0.129,0.048,0.037
1,Net_Ret_LongShort,47.28%**,56.54%***,56.12%***,2.51,3.58,3.46,0.017,0.001,0.002
2,Net_Ret_Modern_XGB_LS,36.35%**,41.21%***,41.30%***,2.19,3.23,3.14,0.036,0.003,0.004
3,Net_Ret_Mom_LS,45.32%***,50.24%***,50.00%***,2.98,3.87,3.74,0.005,0.001,0.001


### 2.3 Carhart 4-Factor Loadings

The Ensemble strategy's full set of factor exposures ("opening the black box") shows what the model's returns are — and aren't — compensating for.

In [6]:
print("Extracting detailed Carhart 4-factor loadings...\n")

strategies_to_analyze_mod = {
    'Modern Ensemble (Long-Short)': 'Net_Ret_LongShort',
    'Modern Ensemble (Long-Only)': 'Net_Ret_LongOnly'
}

def get_stars(pval):
    if pval < 0.01: return '***'
    elif pval < 0.05: return '**'
    elif pval < 0.10: return '*'
    return ''

for name, col in strategies_to_analyze_mod.items():
    print(f"=== {name} FF4 Factor Loadings ===")

    Y = eval_df_mod[col] - eval_df_mod['Rf(1m)']
    X_ff4 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()

    loadings_df = pd.DataFrame({
        'Coef': ff4_model.params,
        't-stat': ff4_model.tvalues,
        'p-value': ff4_model.pvalues
    })
    loadings_df['sig'] = loadings_df['p-value'].apply(get_stars)
    loadings_df.index = ['Alpha (monthly)', 'Market (β_mkt)', 'SMB (β_smb)', 'HML (β_hml)', 'UMD (β_umd)']

    loadings_df['Coef'] = loadings_df['Coef'].map('{:.6f}'.format)
    loadings_df['t-stat'] = loadings_df['t-stat'].map('{:.6f}'.format)
    loadings_df['p-value'] = loadings_df['p-value'].map('{:.6e}'.format)

    print(loadings_df)
    print()

Extracting detailed Carhart 4-factor loadings...

=== Modern Ensemble (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.046767   3.455524  1.661524e-03  ***
Market (β_mkt)    0.288681   0.929363  3.601191e-01     
SMB (β_smb)      -0.428098  -1.608108  1.182865e-01     
HML (β_hml)      -0.534733  -2.441812  2.072430e-02   **
UMD (β_umd)       0.050908   0.159957  8.739872e-01     

=== Modern Ensemble (Long-Only) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.026186   2.181177  3.714188e-02   **
Market (β_mkt)    0.019045   0.069120  9.453528e-01     
SMB (β_smb)      -0.726433  -3.076210  4.445522e-03  ***
HML (β_hml)      -0.226048  -1.163656  2.537353e-01     
UMD (β_umd)      -0.263461  -0.933218  3.581581e-01     



## 3. Historical Track Evaluation

The same three analyses are repeated for the historical track (1980–present), which does not include the Smart Money features.### Historic evaluation

### 3.1 Risk-Adjusted Performance

In [7]:
print("Calculating risk metrics...\n")

risk_results_hist = []

for col in strategy_cols_hist:
    excess_return = eval_df_hist[col] - eval_df_hist['Rf(1m)']

    ann_ret = eval_df_hist[col].mean() * 12
    ann_vol = eval_df_hist[col].std() * np.sqrt(12)

    ann_excess_ret = excess_return.mean() * 12
    sharpe_ratio = ann_excess_ret / ann_vol if ann_vol != 0 else 0

    cum_wealth = (1 + eval_df_hist[col]).cumprod()
    running_max = cum_wealth.cummax()
    drawdowns = (cum_wealth - running_max) / running_max
    max_drawdown = drawdowns.min()

    risk_results_hist.append({
        'Strategy': col,
        'Ann_Return': ann_ret,
        'Ann_Volatility': ann_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown
    })

risk_df_hist = pd.DataFrame(risk_results_hist)
risk_df_hist_display = risk_df_hist.copy()
risk_df_hist_display['Ann_Return'] = risk_df_hist_display['Ann_Return'].map('{:.2%}'.format)
risk_df_hist_display['Ann_Volatility'] = risk_df_hist_display['Ann_Volatility'].map('{:.2%}'.format)
risk_df_hist_display['Sharpe_Ratio'] = risk_df_hist_display['Sharpe_Ratio'].map('{:.2f}'.format)
risk_df_hist_display['Max_Drawdown'] = risk_df_hist_display['Max_Drawdown'].map('{:.2%}'.format)

print("=== Risk-Adjusted Performance Metrics (Historical) ===")
display(risk_df_hist_display)

Calculating risk metrics...

=== Risk-Adjusted Performance Metrics (Historical) ===


,Strategy,Ann_Return,Ann_Volatility,Sharpe_Ratio,Max_Drawdown
0,Net_Ret_LongOnly,33.50%,29.28%,1.03,-8.55%
1,Net_Ret_LongShort,61.49%,29.70%,1.96,-12.91%
2,Net_Ret_Hist_XGB_LS,55.70%,30.21%,1.73,-11.15%
3,Net_Ret_Mom_LS,47.53%,25.73%,1.71,-7.32%
4,VW,8.41%,14.31%,0.35,-12.75%


### 3.2 Alpha Regressions

In [8]:
print("Running CAPM, FF3, and Carhart 4-Factor regressions...\n")

strategies_to_test_hist = [
    'Net_Ret_LongOnly',
    'Net_Ret_LongShort',
    'Net_Ret_Hist_XGB_LS',
    'Net_Ret_Mom_LS'
]

alpha_results_hist = []

for strat in strategies_to_test_hist:
    Y = eval_df_hist[strat] - eval_df_hist['Rf(1m)']

    X_capm = sm.add_constant(eval_df_hist[['Mkt-RF']])
    capm_model = sm.OLS(Y, X_capm).fit()

    X_ff3 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML']])
    ff3_model = sm.OLS(Y, X_ff3).fit()

    X_ff4 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()

    alpha_results_hist.append({
        'Strategy': strat,
        'CAPM_Alpha': capm_model.params['const'] * 12,
        'CAPM_t': capm_model.tvalues['const'],
        'CAPM_p': capm_model.pvalues['const'],

        'FF3_Alpha': ff3_model.params['const'] * 12,
        'FF3_t': ff3_model.tvalues['const'],
        'FF3_p': ff3_model.pvalues['const'],

        'FF4_Alpha': ff4_model.params['const'] * 12,
        'FF4_t': ff4_model.tvalues['const'],
        'FF4_p': ff4_model.pvalues['const']
    })

alpha_df_hist = pd.DataFrame(alpha_results_hist)

for model in ['CAPM', 'FF3', 'FF4']:
    alpha_df_hist[f'{model}_Alpha_Str'] = alpha_df_hist.apply(lambda row: format_alpha(row[f'{model}_Alpha'], row[f'{model}_p']), axis=1)
    alpha_df_hist[f'{model}_t'] = alpha_df_hist[f'{model}_t'].map('{:.2f}'.format)
    alpha_df_hist[f'{model}_p'] = alpha_df_hist[f'{model}_p'].map('{:.3f}'.format)

alpha_table_hist = alpha_df_hist[[
    'Strategy',
    'CAPM_Alpha_Str', 'FF3_Alpha_Str', 'FF4_Alpha_Str',
    'CAPM_t', 'FF3_t', 'FF4_t',
    'CAPM_p', 'FF3_p', 'FF4_p'
]].copy()

alpha_table_hist.columns = [
    'Strategy',
    'CAPM Alpha', 'FF3 Alpha', 'FF4 Alpha',
    'CAPM t-stat', 'FF3 t-stat', 'FF4 t-stat',
    'CAPM p-val', 'FF3 p-val', 'FF4 p-val'
]

print("=== Combined Alpha Summary Table (Historical) ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
display(alpha_table_hist)

Running CAPM, FF3, and Carhart 4-Factor regressions...

=== Combined Alpha Summary Table (Historical) ===
Note: *** p<0.01, ** p<0.05, * p<0.10



,Strategy,CAPM Alpha,FF3 Alpha,FF4 Alpha,CAPM t-stat,FF3 t-stat,FF4 t-stat,CAPM p-val,FF3 p-val,FF4 p-val
0,Net_Ret_LongOnly,31.41%*,29.95%*,32.30%**,1.81,2.01,2.14,0.079,0.053,0.041
1,Net_Ret_LongShort,58.61%***,63.41%***,62.87%***,3.31,4.27,4.11,0.002,0.000,0.000
2,Net_Ret_Hist_XGB_LS,53.09%***,59.39%***,58.12%***,2.95,4.12,3.94,0.006,0.000,0.000
3,Net_Ret_Mom_LS,45.32%***,50.24%***,50.00%***,2.98,3.87,3.74,0.005,0.001,0.001


### 3.3 Carhart 4-Factor Loadings

In [9]:
print("Extracting detailed Carhart 4-factor loadings...\n")

strategies_to_analyze_hist = {
    'Historical Ensemble (Long-Short)': 'Net_Ret_LongShort',
    'Historical Ensemble (Long-Only)': 'Net_Ret_LongOnly',
    'Historical XGB (Long-Short)': 'Net_Ret_Hist_XGB_LS',
}

for name, col in strategies_to_analyze_hist.items():
    print(f"=== {name} FF4 Factor Loadings ===")

    Y = eval_df_hist[col] - eval_df_hist['Rf(1m)']
    X_ff4 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()

    loadings_df = pd.DataFrame({
        'Coef': ff4_model.params,
        't-stat': ff4_model.tvalues,
        'p-value': ff4_model.pvalues
    })
    loadings_df['sig'] = loadings_df['p-value'].apply(get_stars)
    loadings_df.index = ['Alpha (monthly)', 'Market (β_mkt)', 'SMB (β_smb)', 'HML (β_hml)', 'UMD (β_umd)']

    loadings_df['Coef'] = loadings_df['Coef'].map('{:.6f}'.format)
    loadings_df['t-stat'] = loadings_df['t-stat'].map('{:.6f}'.format)
    loadings_df['p-value'] = loadings_df['p-value'].map('{:.6e}'.format)

    print(loadings_df)
    print()

Extracting detailed Carhart 4-factor loadings...

=== Historical Ensemble (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.052389   4.111962  2.804623e-04  ***
Market (β_mkt)    0.234276   0.801184  4.293254e-01     
SMB (β_smb)      -0.535675  -2.137519  4.082304e-02   **
HML (β_hml)      -0.393021  -1.906459  6.620965e-02    *
UMD (β_umd)       0.065449   0.218454  8.285550e-01     

=== Historical Ensemble (Long-Only) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.026917   2.138961  4.069649e-02   **
Market (β_mkt)    0.010692   0.037021  9.707134e-01     
SMB (β_smb)      -0.810639  -3.274991  2.667333e-03  ***
HML (β_hml)      -0.090651  -0.445205  6.593675e-01     
UMD (β_umd)      -0.285595  -0.965114  3.422035e-01     

=== Historical XGB (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.048433   3

## 4. Portfolio Characteristics (Appendix)

For each track, stocks are assigned each month to the Long (top 20%), Short (bottom 20%), or Middle basket based on the Ensemble signal, and their average raw characteristics are computed. This shows *what kind* of stocks the model is actually picking — e.g. whether the Long basket skews toward smaller, higher-momentum, or higher-conviction names.

In [10]:
def assign_quantiles(df_month, signal_col='Ensemble_Pred_z', q=0.20):
    top_cutoff = df_month[signal_col].quantile(1 - q)
    bottom_cutoff = df_month[signal_col].quantile(q)

    conditions = [
        (df_month[signal_col] >= top_cutoff),
        (df_month[signal_col] <= bottom_cutoff)
    ]
    choices = ['Top 20% (Long)', 'Bottom 20% (Short)']

    df_month['Portfolio'] = np.select(conditions, choices, default='Middle 60%')
    return df_month

SORT_ORDER = {'Top 20% (Long)': 1, 'Middle 60%': 2, 'Bottom 20% (Short)': 3}

In [11]:
print("Generating modern-track portfolio characteristics...")

features_df = pd.read_csv('../Data/modern_equity_dataset.csv')
preds_df = pd.read_csv('../Data/modern_predictions.csv')

features_df['TradeDate'] = pd.to_datetime(features_df['TradeDate']) + pd.offsets.MonthEnd(0)
preds_df['TradeDate'] = pd.to_datetime(preds_df['TradeDate']) + pd.offsets.MonthEnd(0)

merged_df = pd.merge(preds_df, features_df, on=['TradeDate', 'ISIN'], how='inner')
merged_df = merged_df.groupby('TradeDate', group_keys=False).apply(assign_quantiles).reset_index()

merged_df

Generating modern-track portfolio characteristics...


,index,ISIN,Target_raw_x,Target_z_x,Ridge_Pred_z,RandomForest_Pred_z,XGBoost_Pred_z,Ensemble_Pred_z,MLP_Pred_z,SecurityType,...,vol_3m_z,vol_6m_z,Target_z_y,BI_Avg_Weight,BI_Fund_Count,BI_Top10_Count,BI_Avg_Weight_z,BI_Fund_Count_z,BI_Top10_Count_z,Portfolio
0,0,AU0000057408,0.055853,0.543014,-0.130330,-0.089271,-0.108623,-0.116870,-0.139258,Ordinary Shares,...,0.696122,0.024718,0.543014,0.0,0.0,0.0,-0.818615,-0.654252,-0.320209,Bottom 20% (Short)
1,1,BMG0451H2087,-0.238462,-1.554438,-0.053540,-0.018204,-0.030865,-0.050227,-0.098299,Ordinary Shares,...,1.053726,0.304838,-1.554438,0.0,0.0,0.0,-0.818615,-0.654252,-0.320209,Middle 60%
2,2,BMG0451H2087,0.030303,-0.266877,-0.010264,-0.014464,-0.006741,-0.009941,-0.008296,Ordinary Shares,...,0.718532,0.510910,-0.266877,0.0,0.0,0.0,-0.778248,-0.652800,-0.306711,Middle 60%
3,3,BMG0451H2087,-0.176471,-1.292151,-0.090075,-0.061955,-0.125239,-0.113640,-0.177289,Ordinary Shares,...,1.084179,0.881926,-1.292151,0.0,0.0,0.0,-0.757330,-0.651486,-0.302484,Bottom 20% (Short)
4,4,BMG0451H2087,0.392857,2.099435,-0.068185,0.036423,0.019929,-0.029365,-0.105628,Ordinary Shares,...,0.812993,0.734226,2.099435,0.0,0.0,0.0,-0.721809,-0.639566,-0.303374,Middle 60%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10289,10289,US36467X2062,-0.035669,-0.003887,-0.012535,0.053839,0.054321,0.026078,0.008687,Ordinary Shares,...,-0.863165,-0.507519,-0.003887,0.0,0.0,0.0,-0.767639,-0.629647,-0.305934,Middle 60%
10290,10290,US36467X2062,-0.107001,-0.828024,0.011562,-0.012629,0.004264,0.010147,0.037392,Ordinary Shares,...,-0.725791,-0.936416,-0.828024,0.0,0.0,0.0,-0.756862,-0.626219,-0.308994,Middle 60%
10291,10291,US36467X2062,-0.013314,0.253301,0.001311,0.044371,0.046911,0.031634,0.033943,Ordinary Shares,...,-0.762047,-1.044621,0.253301,0.0,0.0,0.0,-0.750881,-0.621995,-0.310973,Middle 60%
10292,10292,US36467X2062,0.010495,-0.060196,-0.029112,-0.001031,-0.006990,-0.023461,-0.056711,Ordinary Shares,...,-0.254938,-0.777459,-0.060196,0.0,0.0,0.0,-0.730156,-0.624360,-0.311196,Middle 60%


In [13]:
print("Generating modern-track portfolio characteristics...")

features_df = pd.read_csv('../Data/modern_equity_dataset.csv')
preds_df = pd.read_csv('../Data/modern_predictions.csv')

features_df['TradeDate'] = pd.to_datetime(features_df['TradeDate']) + pd.offsets.MonthEnd(0)
preds_df['TradeDate'] = pd.to_datetime(preds_df['TradeDate']) + pd.offsets.MonthEnd(0)

merged_df = pd.merge(preds_df, features_df, on=['TradeDate', 'ISIN'], how='inner')
merged_df = merged_df.groupby('TradeDate').apply(assign_quantiles).reset_index()

columns_of_interest = ['log_size', 'turnover', 'mom_12m', 'vol_3m', 'BI_Avg_Weight']

monthly_stats = merged_df.groupby(['TradeDate', 'Portfolio'])[columns_of_interest].mean().reset_index()
final_portfolio_stats = monthly_stats.groupby('Portfolio')[columns_of_interest].mean().reset_index()

final_portfolio_stats['Sort_Order'] = final_portfolio_stats['Portfolio'].map(SORT_ORDER)
final_portfolio_stats = final_portfolio_stats.sort_values('Sort_Order').drop(columns=['Sort_Order'])

print("\n=== Average Portfolio Characteristics (Modern Ensemble) ===")
display(final_portfolio_stats)

final_portfolio_stats.to_csv('../Data/appendix_portfolio_characteristics.csv', index=False)

Generating modern-track portfolio characteristics...

=== Average Portfolio Characteristics (Modern Ensemble) ===


,Portfolio,log_size,turnover,mom_12m,vol_3m,BI_Avg_Weight
2,Top 20% (Long),22.098728,0.030531,0.375403,0.088135,1.530643
1,Middle 60%,21.665635,0.033499,0.041820,0.097731,0.860702
0,Bottom 20% (Short),21.335336,0.104736,-0.463677,0.170428,0.312124


In [15]:
print("Generating historical-track portfolio characteristics...")

# investable_universe.csv is the dataset created before adding the modern mutual fund features
features_hist_df = pd.read_csv('../Data/investable_universe.csv')
preds_hist_df = pd.read_csv('../Data/historical_predictions.csv')

features_hist_df['TradeDate'] = pd.to_datetime(features_hist_df['TradeDate']) + pd.offsets.MonthEnd(0)
preds_hist_df['TradeDate'] = pd.to_datetime(preds_hist_df['TradeDate']) + pd.offsets.MonthEnd(0)

merged_hist_df = pd.merge(preds_hist_df, features_hist_df, on=['TradeDate', 'ISIN'], how='inner')
merged_hist_df = merged_hist_df.groupby('TradeDate').apply(assign_quantiles).reset_index()

# BI_Avg_Weight is excluded: mutual fund data isn't part of the historical dataset
columns_of_interest_hist = ['log_size', 'turnover', 'mom_12m', 'vol_3m']

monthly_stats_hist = merged_hist_df.groupby(['TradeDate', 'Portfolio'])[columns_of_interest_hist].mean().reset_index()
final_portfolio_stats_hist = monthly_stats_hist.groupby('Portfolio')[columns_of_interest_hist].mean().reset_index()

final_portfolio_stats_hist['Sort_Order'] = final_portfolio_stats_hist['Portfolio'].map(SORT_ORDER)
final_portfolio_stats_hist = final_portfolio_stats_hist.sort_values('Sort_Order').drop(columns=['Sort_Order'])

print("\n=== Average Portfolio Characteristics (Historical Ensemble) ===")
display(final_portfolio_stats_hist)

final_portfolio_stats_hist.to_csv('../Data/appendix_portfolio_characteristics_hist.csv', index=False)

Generating historical-track portfolio characteristics...

=== Average Portfolio Characteristics (Historical Ensemble) ===


,Portfolio,log_size,turnover,mom_12m,vol_3m
2,Top 20% (Long),21.758076,0.045377,0.492563,0.102635
1,Middle 60%,21.865743,0.033561,0.016599,0.091539
0,Bottom 20% (Short),21.082853,0.089714,-0.505970,0.174291


## 5. Headline Results Summary

A single combined view of risk-adjusted performance and Carhart 4-factor alpha across both tracks, for the flagship comparisons: the ML Ensemble (Long-Short), the XGBoost single-model strategy, the Traditional Momentum baseline, and the Market.

In [16]:
# Combine risk metrics from both tracks into one table, tagged by track
risk_summary = pd.concat([
    risk_df_mod.assign(Track='Modern'),
    risk_df_hist.assign(Track='Historical')
], ignore_index=True)

risk_summary = risk_summary[['Track', 'Strategy', 'Ann_Return', 'Ann_Volatility', 'Sharpe_Ratio', 'Max_Drawdown']]
risk_summary_display = risk_summary.copy()
risk_summary_display['Ann_Return'] = risk_summary_display['Ann_Return'].map('{:.2%}'.format)
risk_summary_display['Ann_Volatility'] = risk_summary_display['Ann_Volatility'].map('{:.2%}'.format)
risk_summary_display['Sharpe_Ratio'] = risk_summary_display['Sharpe_Ratio'].map('{:.2f}'.format)
risk_summary_display['Max_Drawdown'] = risk_summary_display['Max_Drawdown'].map('{:.2%}'.format)

print("=== Combined Risk-Adjusted Performance (Modern vs. Historical) ===")
display(risk_summary_display)

# Combine alpha regression results from both tracks into one table, tagged by track
alpha_summary = pd.concat([
    alpha_table_mod.assign(Track='Modern'),
    alpha_table_hist.assign(Track='Historical')
], ignore_index=True)

alpha_summary = alpha_summary[['Track'] + [c for c in alpha_summary.columns if c != 'Track']]

print("\n=== Combined Alpha Summary (Modern vs. Historical) ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
display(alpha_summary)

risk_summary.to_csv('../Data/headline_risk_summary.csv', index=False)
alpha_summary.to_csv('../Data/headline_alpha_summary.csv', index=False)

=== Combined Risk-Adjusted Performance (Modern vs. Historical) ===


,Track,Strategy,Ann_Return,Ann_Volatility,Sharpe_Ratio,Max_Drawdown
0,Modern,Net_Ret_LongOnly,28.87%,29.30%,0.87,-9.14%
1,Modern,Net_Ret_LongShort,50.26%,31.64%,1.48,-22.30%
2,Modern,Net_Ret_Modern_XGB_LS,39.65%,27.81%,1.30,-11.36%
3,Modern,Net_Ret_Mom_LS,47.53%,25.73%,1.71,-7.32%
4,Modern,VW,8.41%,14.31%,0.35,-12.75%
5,Historical,Net_Ret_LongOnly,33.50%,29.28%,1.03,-8.55%
6,Historical,Net_Ret_LongShort,61.49%,29.70%,1.96,-12.91%
7,Historical,Net_Ret_Hist_XGB_LS,55.70%,30.21%,1.73,-11.15%
8,Historical,Net_Ret_Mom_LS,47.53%,25.73%,1.71,-7.32%
9,Historical,VW,8.41%,14.31%,0.35,-12.75%



=== Combined Alpha Summary (Modern vs. Historical) ===
Note: *** p<0.01, ** p<0.05, * p<0.10



,Track,Strategy,CAPM Alpha,FF3 Alpha,FF4 Alpha,CAPM t-stat,FF3 t-stat,FF4 t-stat,CAPM p-val,FF3 p-val,FF4 p-val
0,Modern,Net_Ret_LongOnly,26.94%,29.25%**,31.42%**,1.56,2.06,2.18,0.129,0.048,0.037
1,Modern,Net_Ret_LongShort,47.28%**,56.54%***,56.12%***,2.51,3.58,3.46,0.017,0.001,0.002
2,Modern,Net_Ret_Modern_XGB_LS,36.35%**,41.21%***,41.30%***,2.19,3.23,3.14,0.036,0.003,0.004
3,Modern,Net_Ret_Mom_LS,45.32%***,50.24%***,50.00%***,2.98,3.87,3.74,0.005,0.001,0.001
4,Historical,Net_Ret_LongOnly,31.41%*,29.95%*,32.30%**,1.81,2.01,2.14,0.079,0.053,0.041
5,Historical,Net_Ret_LongShort,58.61%***,63.41%***,62.87%***,3.31,4.27,4.11,0.002,0.000,0.000
6,Historical,Net_Ret_Hist_XGB_LS,53.09%***,59.39%***,58.12%***,2.95,4.12,3.94,0.006,0.000,0.000
7,Historical,Net_Ret_Mom_LS,45.32%***,50.24%***,50.00%***,2.98,3.87,3.74,0.005,0.001,0.001
